In [0]:
repos = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_repo_data
        WHERE id IS NULL 
        OR id <= 0
        OR name IS NULL 
        OR TRIM(name) != name 
        OR full_name IS NULL 
        OR TRIM(full_name) != full_name
        OR org_login IS NULL
        OR TRIM(org_login) != org_login
        OR description IS NULL
        OR TRIM(description) != description
        OR language IS NULL
        OR TRIM(language) != language
        OR created_at IS NULL
        OR private IS NULL
        OR fork IS NULL
        OR is_template IS NULL
        OR allow_forking IS NULL
        OR has_issues IS NULL
        OR has_pull_requests IS NULL
        OR has_discussions IS NULL
        OR has_wiki IS NULL
        OR has_pages IS NULL
        OR has_downloads IS NULL
        OR pull_request_creation_policy IS NULL 
        OR TRIM(pull_request_creation_policy) != pull_request_creation_policy
        OR name = '' OR full_name = '' OR org_login = ''
        OR description = '' OR language = '' OR pull_request_creation_policy = ''
              """).display()
    

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT * 
        FROM workspace.silver.{repo}_contributors
        WHERE id IS NULL
        OR id <= 0
        OR login IS NULL
        OR TRIM(login) != login
        OR login = ''
        OR type IS NULL 
        OR TRIM(type) != type
        OR type = ''
        OR contributions IS NULL
        OR site_admin IS NULL
        OR repo_name IS NULL
              """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT id, COUNT(*) AS n
        FROM workspace.silver.{repo}_contributors
        GROUP BY id
        HAVING COUNT(*) > 1
    """).display()

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT * 
        FROM workspace.silver.{repo}_languages
        WHERE language IS NULL
        OR TRIM(language) != language
        OR language = ''
        OR bytes IS NULL
        OR repo_name IS NULL
              """).display()

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT * 
        FROM workspace.silver.{repo}_pr_assignees
        WHERE pr_id IS NULL 
        OR pr_id <= 0
        OR assignee_id IS NULL
        OR assignee_id <= 0
        OR assignee_login IS NULL
        OR TRIM(assignee_login) != assignee_login
        OR assignee_login = ''
        OR repo_name IS NULL
        OR TRIM(repo_name) != repo_name
        OR repo_name = ''
              """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pr_assignees
        WHERE assignee_type IS NULL
           OR TRIM(assignee_type) != assignee_type
           OR assignee_type = ''
           OR (assignee_id IS NULL AND (assignee_id_is_missing != 1
                                        OR assignee_login != 'n/a'
                                        OR assignee_type != 'n/a'))
           OR (assignee_id IS NOT NULL AND (assignee_id_is_missing != 0
                                            OR assignee_login = 'n/a'))
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT pr_id, assignee_id, COUNT(*) AS n
        FROM workspace.silver.{repo}_pr_assignees
        GROUP BY pr_id, assignee_id
        HAVING COUNT(*) > 1
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pr_reviewers
        WHERE pr_id IS NULL
        OR pr_id <= 0
        OR reviewer_id IS NULL
        OR reviewer_id <= 0
        OR reviewer_login IS NULL
        OR TRIM(reviewer_login) != reviewer_login
        OR reviewer_login = ''
        OR repo_name IS NULL
        OR TRIM(repo_name) != repo_name
        OR repo_name = ''
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pr_reviewers
        WHERE reviewer_type IS NULL
        OR TRIM(reviewer_type) != reviewer_type
        OR reviewer_type = ''
        OR (reviewer_id IS NULL AND (reviewer_id_is_missing != 1
                                     OR reviewer_login != 'n/a'
                                     OR reviewer_type != 'n/a'))
        OR (reviewer_id IS NOT NULL AND (reviewer_id_is_missing != 0
                                         OR reviewer_login = 'n/a'))
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT pr_id, reviewer_id, COUNT(*) AS n
        FROM workspace.silver.{repo}_pr_reviewers
        GROUP BY pr_id, reviewer_id
        HAVING COUNT(*) > 1
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pr_labels
        WHERE pr_id IS NULL
        OR pr_id <= 0
        OR label_id IS NULL
        OR label_id <= 0
        OR label_name IS NULL
        OR TRIM(label_name) != label_name
        OR label_name = ''
        OR repo_name IS NULL
        OR TRIM(repo_name) != repo_name
        OR repo_name = ''
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pr_labels
        WHERE label_color IS NULL
        OR TRIM(label_color) != label_color
        OR label_color = ''
        OR (label_id IS NULL AND (label_id_is_missing != 1
                                  OR label_name != 'n/a'
                                  OR label_color != 'n/a'))
        OR (label_id IS NOT NULL AND (label_id_is_missing != 0
                                      OR label_name = 'n/a'
                                      OR label_color = 'n/a'))
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT pr_id, label_id, COUNT(*) AS n
        FROM workspace.silver.{repo}_pr_labels
        GROUP BY pr_id, label_id
        HAVING COUNT(*) > 1
    """).display()

In [0]:
str_cols = ["title", "state", "body", "author_association", "user_login", "user_type",
            "merge_commit_sha", "base_ref", "base_sha", "base_repo_full_name",
            "head_ref", "head_sha", "head_repo_full_name",
            "milestone_title", "milestone_state", "auto_merge_method",
            "repo_name", "timestamp_issue"]

str_conditions = "\n OR ".join(
    f"{c} IS NULL OR TRIM({c}) != {c} OR {c} = ''" for c in str_cols
)

for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pull_requests
        WHERE pr_id IS NULL OR pr_id <= 0
           OR number IS NULL OR number <= 0
           OR created_at IS NULL OR updated_at IS NULL
           OR draft IS NULL OR locked IS NULL
           OR user_id <= 0 OR base_repo_id <= 0 OR head_repo_id <= 0 OR milestone_id <= 0
           OR {str_conditions}
    """).display()

In [0]:
flag_cols = ["pr_id", "number", "created_at", "updated_at", "draft", "locked",
             "user_id", "user_site_admin", "closed_at", "merged_at",
             "base_repo_id", "head_repo_id", "milestone_id"]

flag_conditions = "\n OR ".join(
    f"({c} IS NULL) != ({c}_is_missing = 1)" for c in flag_cols
)

for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_pull_requests
        WHERE {flag_conditions}
    """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT pr_id, COUNT(*) AS n
        FROM workspace.silver.{repo}_pull_requests
        GROUP BY pr_id
        HAVING COUNT(*) > 1
    """).display()

In [0]:
date_problem = """(created_at > updated_at OR closed_at < created_at
                   OR merged_at < created_at OR merged_at > closed_at)"""

for repo in repos:
    spark.sql(f"""
        SELECT pr_id, created_at, updated_at, closed_at, merged_at, timestamp_issue
        FROM workspace.silver.{repo}_pull_requests
        WHERE (timestamp_issue = 'n/a' AND {date_problem})
           OR (timestamp_issue != 'n/a' AND NOT COALESCE({date_problem}, FALSE))
    """).display()

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_commits
        WHERE sha IS NULL
        OR sha = ''
        OR TRIM(sha) != sha
        OR commit_message IS NULL 
        OR commit_message = ''
        OR TRIM(commit_message) != commit_message
        OR commit_author_date IS NULL
        OR commit_author_date < '2000-01-01'
        OR commit_author_name IS NULL
        OR commit_author_name = ''
        OR TRIM(commit_author_name) != commit_author_name
        OR commit_author_email IS NULL
        OR commit_author_email = ''
        OR TRIM(commit_author_email) != commit_author_email
        OR commit_committer_date IS NULL
        OR commit_committer_date < '2000-01-01'
        OR commit_committer_name IS NULL
        OR commit_committer_name = ''
        OR TRIM(commit_committer_name) != commit_committer_name
        OR commit_committer_email IS NULL
        OR commit_committer_email = ''
        OR TRIM(commit_committer_email) != commit_committer_email
        OR author_id <= 0
        OR author_login IS NULL
        OR author_login = ''
        OR TRIM(author_login) != author_login
        OR author_type IS NULL
        OR TRIM(author_type) != author_type
        OR author_type = ''
        OR committer_id <= 0
        OR committer_login IS NULL
        OR committer_login = ''
        OR TRIM(committer_login) != committer_login
        OR committer_type IS NULL
        OR TRIM(committer_type) != committer_type
        OR committer_type = ''
        OR repo_name IS NULL
        OR TRIM(repo_name) != repo_name
        OR repo_name = ''
              """).display()

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT sha, COUNT(*)
        FROM workspace.silver.{repo}_commits
        GROUP BY sha
        HAVING COUNT(*) > 1
              """).display()

In [0]:
date_problem = """(commit_author_date > commit_committer_date OR commit_author_date < '2000-01-01'
                   OR commit_committer_date < '2000-01-01')"""

for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_commits
        WHERE commit_author_date < '2000-01-01'
                   OR commit_committer_date < '2000-01-01'
    """).display()

In [0]:
flag_cols = ["sha", "commit_author_date", "commit_committer_date", "comment_count",
             "author_id", "author_site_admin", "committer_id", "committer_site_admin"]

flag_conditions = "\n OR ".join(
    f"({c} IS NULL) != ({c}_is_missing = 1)" for c in flag_cols
)

for repo in repos:
    spark.sql(f"""
        SELECT *
        FROM workspace.silver.{repo}_commits
        WHERE {flag_conditions}
    """).display()

In [0]:
%sql
SELECT user_login, COUNT(*) AS n
FROM silver.vscode_pull_requests
WHERE user_login != 'n/a'
  AND NOT user_login RLIKE '^[A-Za-z0-9]([A-Za-z0-9-]{0,38})(\\[bot\\])?$'
GROUP BY user_login ORDER BY n DESC;

In [0]:
%sql
SELECT COUNT(*) FROM silver.vscode_pull_requests
WHERE user_login RLIKE '\\s' OR user_login RLIKE '[\\x00-\\x1F\\x7F]';

In [0]:
%sql
SELECT COUNT(*) AS bad_emails
FROM silver.vscode_commits
WHERE commit_author_email != 'n/a'
  AND NOT commit_author_email RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$';

In [0]:
%sql
SELECT commit_author_email, COUNT(*) AS n
FROM silver.vscode_commits
WHERE commit_author_email != 'n/a'
  AND NOT commit_author_email RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$'
GROUP BY commit_author_email
ORDER BY n DESC;

In [0]:
%sql
WITH all_commits AS (
  SELECT 'vscode' AS repo, commit_author_email FROM silver.vscode_commits UNION ALL
  SELECT 'aspnetcore', commit_author_email FROM silver.aspnetcore_commits UNION ALL
  SELECT 'powershell', commit_author_email FROM silver.powershell_commits UNION ALL
  SELECT 'powertoys',  commit_author_email FROM silver.powertoys_commits UNION ALL
  SELECT 'runtime',    commit_author_email FROM silver.runtime_commits UNION ALL
  SELECT 'terminal',   commit_author_email FROM silver.terminal_commits UNION ALL
  SELECT 'typescript', commit_author_email FROM silver.typescript_commits
),
flagged AS (
  SELECT repo,
         CASE WHEN commit_author_email != 'n/a'
               AND NOT commit_author_email RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$'
              THEN 1 ELSE 0 END AS is_bad
  FROM all_commits
)
SELECT COALESCE(repo, 'TOTAL') AS repo,
       COUNT(*)                          AS total_commits,
       SUM(is_bad)                       AS bad_emails,
       ROUND(100.0 * SUM(is_bad) / COUNT(*), 4) AS pct_bad
FROM flagged
GROUP BY ROLLUP (repo)
ORDER BY (repo IS NULL), pct_bad DESC;

In [0]:
%sql
WITH logins AS (
  SELECT 'vscode' AS repo, user_login AS login FROM silver.vscode_pull_requests UNION ALL
  SELECT 'aspnetcore', user_login FROM silver.aspnetcore_pull_requests UNION ALL
  SELECT 'powershell', user_login FROM silver.powershell_pull_requests UNION ALL
  SELECT 'powertoys',  user_login FROM silver.powertoys_pull_requests UNION ALL
  SELECT 'runtime',    user_login FROM silver.runtime_pull_requests UNION ALL
  SELECT 'terminal',   user_login FROM silver.terminal_pull_requests UNION ALL
  SELECT 'typescript', user_login FROM silver.typescript_pull_requests
)
SELECT COALESCE(repo, 'TOTAL') AS repo,
       COUNT(*) AS total_rows,
       SUM(CASE WHEN login = 'n/a' THEN 1 ELSE 0 END) AS placeholder,
       SUM(CASE WHEN login != 'n/a'
                 AND NOT login RLIKE '^[A-Za-z0-9]([A-Za-z0-9-]{0,38})(\\[bot\\])?$'
                THEN 1 ELSE 0 END) AS bad_format,
       SUM(CASE WHEN login != TRIM(login) OR login RLIKE '\\s' THEN 1 ELSE 0 END) AS has_whitespace
FROM logins
GROUP BY ROLLUP (repo)
ORDER BY (repo IS NULL), repo;

In [0]:
%sql
SELECT user_id, COUNT(DISTINCT user_login) AS logins, COLLECT_SET(user_login) AS names
FROM silver.vscode_pull_requests
WHERE user_login != 'n/a'
GROUP BY user_id HAVING COUNT(DISTINCT user_login) > 1;

In [0]:
%sql
SELECT user_login, COUNT(DISTINCT user_id) AS ids, COLLECT_SET(user_id) AS id_list
FROM silver.vscode_pull_requests
WHERE user_login != 'n/a'
GROUP BY user_login HAVING COUNT(DISTINCT user_id) > 1;

In [0]:
%sql
SELECT LOWER(user_login) AS login_lower,
       COUNT(DISTINCT user_login) AS variants,
       COLLECT_SET(user_login) AS spellings
FROM silver.vscode_pull_requests
WHERE user_login != 'n/a'
GROUP BY LOWER(user_login) HAVING COUNT(DISTINCT user_login) > 1;

In [0]:
%sql
SELECT 'assignees' AS tbl, COUNT(*) AS orphans
FROM silver.vscode_pr_assignees a LEFT ANTI JOIN silver.vscode_pull_requests p ON a.pr_id = p.pr_id
UNION ALL
SELECT 'reviewers', COUNT(*)
FROM silver.vscode_pr_reviewers a LEFT ANTI JOIN silver.vscode_pull_requests p ON a.pr_id = p.pr_id
UNION ALL
SELECT 'labels', COUNT(*)
FROM silver.vscode_pr_labels a LEFT ANTI JOIN silver.vscode_pull_requests p ON a.pr_id = p.pr_id;

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT 'assignees' AS tbl, COUNT(*) AS orphans
        FROM silver.{repo}_pr_assignees a LEFT ANTI JOIN silver.{repo}_pull_requests p ON a.pr_id = p.pr_id
        UNION ALL
        SELECT 'reviewers', COUNT(*)
        FROM silver.{repo}_pr_reviewers a LEFT ANTI JOIN silver.{repo}_pull_requests p ON a.pr_id = p.pr_id
        UNION ALL
        SELECT 'labels', COUNT(*)
        FROM silver.{repo}_pr_labels a LEFT ANTI JOIN silver.{repo}_pull_requests p ON a.pr_id = p.pr_id;
              """).display()

In [0]:
for repo in repos :
    spark.sql(f"""
        SELECT pr_id, assignee_id, COUNT(*) c
        FROM silver.{repo}_pr_assignees GROUP BY pr_id, assignee_id HAVING COUNT(*) > 1;
              """).display()

In [0]:
for repo in repos:
    spark.sql(f"""
        SELECT pr_id,
            SUM(CASE WHEN assignee_id_is_missing = 1 THEN 1 ELSE 0 END) AS placeholder_rows,
            SUM(CASE WHEN assignee_id_is_missing = 0 THEN 1 ELSE 0 END) AS real_rows
        FROM silver.vscode_pr_assignees
        GROUP BY pr_id
        HAVING SUM(CASE WHEN assignee_id_is_missing = 1 THEN 1 ELSE 0 END) > 0
        AND SUM(CASE WHEN assignee_id_is_missing = 0 THEN 1 ELSE 0 END) > 0;
              """).display()